In [21]:
from pathlib import Path
import re
import zipfile
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

root = Path(".")
archive = root / "campaign_success.zip"
data_dir = root / "data"

if archive.exists():
    data_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(data_dir)

train_path = next(root.rglob("train.csv"))
test_path = next(root.rglob("test.csv"))

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train.shape, test.shape

((27465, 32), (7000, 31))

## Подготовка

In [22]:
target = "campaign_success"

X = train.drop(columns=target).copy()
T = test.copy()

le = LabelEncoder()
y = le.fit_transform(train[target])
assert len(le.classes_) == 2

for c in X.columns:
    if X[c].dtype == "object":
        s = pd.concat([X[c], T[c]], ignore_index=True)
        z = pd.to_numeric(s, errors="coerce")
        if z.notna().mean() > 0.98:
            X[c] = pd.to_numeric(X[c], errors="coerce")
            T[c] = pd.to_numeric(T[c], errors="coerce")

id_cols = [
    c for c in X.columns
    if re.search(r"(^id$|_id$|^index$|uuid)", c, re.I)
    and X[c].nunique(dropna=False) > 0.5 * len(X)
]

X = X.drop(columns=id_cols)
T = T.drop(columns=id_cols)

X["_missing"] = X.isna().sum(axis=1)
T["_missing"] = T.isna().sum(axis=1)

X = X.replace([np.inf, -np.inf], np.nan)
T = T.replace([np.inf, -np.inf], np.nan)

cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

for c in cat_cols:
    X[c] = X[c].fillna("__NA__").astype(str)
    T[c] = T[c].fillna("__NA__").astype(str)

len(X), X.shape[1], len(cat_cols), np.bincount(y)

(27465, 32, 4, array([17869,  9596]))

In [23]:
id_cols

[]

## Модель

In [ ]:
folds = list(StratifiedKFold(4, shuffle=True, random_state=42).split(X, y))

configs = [
    {"depth": 6, "l2_leaf_reg": 5},
    {"depth": 7, "l2_leaf_reg": 8, "auto_class_weights": "Balanced"},
    {"depth": 8, "l2_leaf_reg": 8}
]

results = []

for i, extra in enumerate(configs):
    oof = np.zeros(len(X))
    pred = np.zeros(len(T))

    for tr_idx, va_idx in folds:
        model = CatBoostClassifier(
            iterations=1600,
            learning_rate=0.04,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=2026 + i,
            random_strength=0.5,
            verbose=False,
            allow_writing_files=False,
            **extra
        )

        model.fit(
            X.iloc[tr_idx],
            y[tr_idx],
            cat_features=cat_cols,
            eval_set=(X.iloc[va_idx], y[va_idx]),
            early_stopping_rounds=120,
            verbose=False
        )

        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        pred += model.predict_proba(T)[:, 1] / len(folds)

    grid = np.unique(np.quantile(oof, np.linspace(0.001, 0.999, 501)))
    scores = np.array([f1_score(y, oof >= t, average="macro") for t in grid])
    j = scores.argmax()

    results.append({
        "score": scores[j],
        "threshold": grid[j],
        "pred": pred,
        "params": extra
    })

[(round(r["score"], 5), round(r["threshold"], 4), r["params"]) for r in results]

## Сабмит

In [ ]:
best = max(results, key=lambda r: r["score"])

labels = le.inverse_transform((best["pred"] >= best["threshold"]).astype(int))
submission = pd.DataFrame({target: labels})
submission.to_csv("submission.csv", index=False)

best["score"], best["threshold"], submission[target].value_counts(), submission.head()